> [!WARNING]
> **METADATA BLOCKED SUBMISSION WORKFLOW**:
> Official Kaggle `test.csv` rows contain only `sample_id`, `image_path`, and `target_name`. They lack all training metadata (`State`, `Species`, `NDVI`, `Height`, `Sampling_Date`). This model cannot generate standard test predictions without metadata. See `notebooks/evaluation/unified_metadata_evaluation.ipynb` for counterfactual evaluation.


> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK**
> Internet is explicitly disabled in this notebook as per Kaggle requirements.
> - You MUST upload your pre-trained fold checkpoints (e.g. `fold0_best.pth`) as a Kaggle dataset and point `CFG.MODEL_DIR` to it.
> - You MUST upload the backbone weights (timm weights) as a dataset if the model requires them, or use a pre-downloaded hub folder.


# A2: Metadata Ablation (With vs. Without Metadata)

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

This ablation quantifies the contribution of the metadata MLP stream. The experiment is run once with metadata fusion enabled and once with image-only features to measure the performance delta.

### Training Protocol
- **Seed:** 17 (deterministic)
- **Epochs:** 50 (max)
- **Early Stopping Patience:** 10 epochs


## Inference and Submission Generation (A2).

Load the best checkpoint from each fold, run inference on test images,
average predictions across folds, and produce the submission CSV.


In [ ]:
# --- Fail Fast Schema Check for Official Kaggle Test Data ---
import pandas as pd
import os

test_csv_candidates = [
    '/kaggle/input/csiro-biomass/test.csv',
    '/kaggle/input/competitions/csiro-biomass/test.csv',
    'csiro-biomass/test.csv',
]
test_csv_path = next((p for p in test_csv_candidates if os.path.isfile(p)), None)
if test_csv_path is not None:
    test_df = pd.read_csv(test_csv_path)
    required_meta = ['State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'Sampling_Date']
    missing = [c for c in required_meta if c not in test_df.columns]
    if missing:
        raise ValueError(
            f"Official test.csv lacks required metadata columns: {missing}. "
            "This model was trained with 23 tabular metadata features and is metadata blocked "
            "by the official competition test schema. Do not silently fill zeros or fabricate metadata. "
            "See notebooks/evaluation/unified_metadata_evaluation.ipynb for zero metadata counterfactual evaluation."
        )


In [ ]:
# --- STEP 3: Inference and Submission (A2) ---

class TestBiomassDataset(Dataset):
    """Test dataset: returns left/right image halves (no labels)."""
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.basename(self.paths[idx])
        path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        mid = w // 2
        left = img[:, :mid]
        right = img[:, mid:]
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        return left, right

@torch.no_grad()
def predict_test(model, loader, device):
    """Run inference and return predictions array."""
    model.eval()
    all_preds = []
    for left, right in tqdm(loader, desc='Inference'):
        left = left.to(device)
        right = right.to(device)
        with autocast():
            preds = model(left, right)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

# Build test loader using unique images (same as test_df from training).
test_dataset = TestBiomassDataset(test_df, CFG.TEST_IMAGE_DIR, get_val_transforms())
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

# Ensemble predictions across all trained folds.
all_fold_preds = []
for fold in CFG.FOLDS_TO_TRAIN:
    ckpt_path = f'{CFG.MODEL_DIR}/fold{fold}_best.pth'
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path}, skipping fold {fold}.')
        continue
    model = BiomassModel(CFG.MODEL_NAME, pretrained=False).to(CFG.DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE))
    print(f'Loaded fold {fold} checkpoint.')
    fold_preds = predict_test(model, test_loader, CFG.DEVICE)
    all_fold_preds.append(fold_preds)
    del model
    gc.collect()
    torch.cuda.empty_cache()

if len(all_fold_preds) == 0:
    raise RuntimeError('No fold checkpoints found. Cannot generate submission.')

avg_preds = np.mean(all_fold_preds, axis=0)
print(f'Ensemble of {len(all_fold_preds)} folds. Predictions shape: {avg_preds.shape}')


In [ ]:
# --- Build submission CSV (A2) ---
# Model outputs 5 columns: [Dry_Green_g, Dry_Dead_g, Dry_Clover_g, GDM_g, Dry_Total_g]
# We must map these back to the sample_ids from test.csv.

# Read the original test.csv (long format with sample_id per target).
test_long = pd.read_csv(CFG.TEST_CSV)

# Get the unique image IDs in the same order as test_df (used for inference).
image_ids = test_df['image_id'].values  # unique images, same order as predictions

# Create a mapping from (image_id, target_name) -> predicted value.
pred_map = {}
for i, img_id in enumerate(image_ids):
    for j, target_name in enumerate(CFG.TARGET_COLS):
        pred_map[(img_id, target_name)] = float(avg_preds[i, j])

# Map predictions to the test.csv sample_ids.
test_long['image_id'] = test_long['sample_id'].str.split('__').str[0]
test_long['target'] = test_long.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1
)

# Build final submission: only sample_id and target columns.
df_sub = test_long[['sample_id', 'target']].copy()

# Ensure correct ordering from sample_submission.csv.
sample_sub = pd.read_csv(os.path.join(CFG.BASE_PATH, 'sample_submission.csv'))
df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
df_sub['target'] = df_sub['target'].fillna(0.0)

df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv')
print(f'Shape: {df_sub.shape}')
print(df_sub.to_string())
